# Stage 0: is InceptionV1 `mixed4e:55` one V-shape detector?

Independent test of a claim in Goodfire's *Neural Networks Are Already Clean*: the classic
"cat faces + car fronts" polysemantic neuron is really a single pointed-front / V detector
whose negative pole is round shapes.

**Runtime:** Runtime → Change runtime type → **A100 GPU** (T4 also works, just slower).
The whole notebook takes well under an hour of GPU time; most of it is downloading ImageNet val.

**Before you start (once):**
1. On huggingface.co, open the `ILSVRC/imagenet-1k` dataset page and accept the terms.
2. Create a *read* token; in Colab click the key icon (Secrets), add `HF_TOKEN`, enable notebook access.
3. Put this repo in Google Drive at `MyDrive/already-clean-check` (or push it to GitHub and set `REPO_URL`).

Order matters: the preregistered test runs **before** you look at any example images.

In [ ]:
# ---- config ----
SMOKE = False   # True: tiny synthetic data, runs anywhere (pipeline test only, results meaningless)
REPO_URL = "https://github.com/TheoHLong/already-clean-check.git"   # used only if DRIVE_CODE is missing
DRIVE_CODE = "/content/drive/MyDrive/already-clean-check"
RUN_ROOT = "/content/drive/MyDrive/acc_runs/stage0"   # cache + figures survive disconnects
UNIT = "mixed4e:55"
LAYERS = ["mixed4d", "mixed4e", "mixed5a"]            # inputs, the unit, its readers
BATCH = 256

In [ ]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CODE = DRIVE_CODE if os.path.isdir(DRIVE_CODE) else "/content/already-clean-check"
    if not os.path.isdir(CODE):
        subprocess.run(["git", "clone", "-q", REPO_URL, CODE], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "torch-lucent==0.1.8", "datasets", "huggingface_hub"], check=True)
else:
    CODE = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if SMOKE:
    RUN_ROOT = os.path.join("/content" if IN_COLAB else CODE, "smoke_run")
sys.path.insert(0, CODE)
os.makedirs(RUN_ROOT, exist_ok=True)

import torch, matplotlib.pyplot as plt, pandas as pd
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("code:", CODE, "| runs:", RUN_ROOT, "| device:", DEVICE,
      torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## 1. ImageNet validation set (50k images)
Downloaded to the Colab VM's local disk (fast); redo this cell in each new session.

In [ ]:
from acc.data import download_hf_val, open_val, make_fake_imagefolder
if SMOKE:
    VAL_DIR = make_fake_imagefolder(os.path.join(RUN_ROOT, "fake_val"))
else:
    if IN_COLAB:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    VAL_DIR = download_hf_val("/content/imagenet_val")
source = open_val(VAL_DIR)
print(type(source).__name__, len(source), "images")

## 2. Model and circuit sanity check
Original Lucid InceptionV1 weights. If indexing matches the essay, `mixed4d:493` should be the #1 input and `mixed5a:94` / `mixed5a:352` the #1 readers.

In [ ]:
from acc.models import load_inceptionv1
from acc.stage0 import circuit_check, parse_unit
model = load_inceptionv1(DEVICE)
LAYER, CH = parse_unit(UNIT)
circ = circuit_check(model, UNIT)
display(circ["essay_connections"])
display(circ["inputs"].head(8))
display(circ["readers"].head(8))

## 3. Activation cache
Pre-ReLU spatial max/min for every image × channel. Written to Drive in shards; if Colab disconnects, rerun and it resumes.

In [ ]:
from acc.cache import cache_activations, load_cache
CACHE = os.path.join(RUN_ROOT, "cache_inceptionv1")
cache_activations(model, source, LAYERS, CACHE, batch_size=BATCH, num_workers=os.cpu_count(),
                  device=DEVICE, shard_size=16 if SMOKE else 5000)
caches = {l: load_cache(CACHE, l) for l in LAYERS}
c = caches[LAYER]
print({l: v["max"].shape for l, v in caches.items()})

## 4. Preregistered test (run BEFORE looking at examples)

**Stop here first.** Edit `prereg/4e55.json` and `PREREGISTRATION.md` with your own predictions,
then `git commit` (and push) them. Only then run the cell below. Class names must match
`acc.data.CATEGORIES` exactly.

In [ ]:
from acc import analysis as A
PREREG = os.path.join(CODE, "prereg", "4e55.json")
prereg_df, prereg_summary = A.prereg_test(c, CH, PREREG)
display(prereg_df)
prereg_summary

## 5. Dataset examples: positive and negative poles
Full images, then 112 px crops centred on where the unit peaked.

In [ ]:
p99 = A.percentile_scale(c, CH)
print(f"natural-image p99 of spatial max: {p99:.2f}")
for pole in ("pos", "neg"):
    for crop in (None, 112):
        A.show_examples(source, c, CH, k=18, pole=pole, crop=crop); plt.show()
    comp = A.class_composition(c, CH, n=100, pole=pole)
    print(pole, comp.attrs)
    display(comp.head(15))

## 6. Synthetic shapes (a tuning-curve experiment)
Responses are divided by the natural-image p99, so 1.0 means as strong as a top-1% natural image.

In [ ]:
from acc.stage0 import synthetic_tuning, plot_tuning
tun = synthetic_tuning(model, UNIT, DEVICE, natural_p99=p99)
plot_tuning(tun, UNIT); plt.show()
tun["battery"].groupby(["kind", "polarity"])["response_norm"].mean().unstack().round(2)

## 7. What feeds it and what reads it
Top 3 inputs (mixed4d) and readers (mixed5a) by weight, each with its own strongest crops.

In [ ]:
from acc.circuits import input_weights, reader_weights
for df_, col in [(input_weights(model, LAYER, CH, top=3), "from"), (reader_weights(model, LAYER, CH, top=3), "to")]:
    for name, w in zip(df_[col], df_["weight"]):
        l, ch = name.split(":")
        if l in caches:
            A.show_examples(source, caches[l], int(ch), k=6, crop=112, title=f"{name}  (weight {w:+.3f})")
            plt.show()

## 8. Feature visualisation (positive and negative direction)

In [ ]:
from acc.stage0 import feature_vis
steps, size = (8, 64) if SMOKE else (512, 224)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, neg in zip(axes, (False, True)):
    ax.imshow(feature_vis(UNIT, negative=neg, steps=steps, size=size))
    ax.set_title("negative" if neg else "positive"); ax.axis("off")
plt.show()

## 9. Save everything to Drive
CSV tables, figures and `summary.json` go to `RUN_ROOT/mixed4e_55/`.

In [ ]:
from acc.stage0 import run_all
summary = run_all(model, source, CACHE, UNIT, PREREG, os.path.join(RUN_ROOT, UNIT.replace(":", "_")), device=DEVICE)
{k: v for k, v in summary.items() if k != "essay_connections"}

## Next
* Swap `UNIT` for any other channel (e.g. the readers above) and rerun from section 4.
* When Colab compute runs out: `modal run modal_app.py --step download`, then `--step stage0` (see README).